<a href="https://colab.research.google.com/github/Ut0p4nPr0c3ssingUnit/CTA-Chartered-Tax-Advisor-AI-Revision-Tool-Proof-of-concept/blob/main/Chartered_Tax_Advisor_(CTA)_Agentic_Model_Producing_Flashcards_based_on_Tolley's_Material.ipynb">Run in Google Colab</a>

**Installing the relevant Python Libraries and Packages**

In [ ]:
# SYSTEM SETUP: DEPENDENCIES & AUTHORISATION
import os
import json
import google.auth
from google.colab import auth
from google.cloud import storage
import vertexai
from vertexai.generative_models import GenerativeModel, Part, GenerationConfig
import IPython

print("\n[System] Initialising Google Cloud Account Authentication...")
auth.authenticate_user()

**Automatically finding the current Project ID in Google Cloud Colab Coding Platform**

In [ ]:
# AUTOMATED ID DETECTION: The notebook securely extracts your project identifier on launch
try:
    credentials, project_id = google.auth.default()
    if not project_id:
        raise ValueError("Project ID could not be detected. Please ensure your GCP billing console is active.")
    PROJECT_ID = project_id
    print(f"[System] Successfully auto-detected active Project ID: {PROJECT_ID}")
except Exception as e:
    print(f"[System Error] Automatic project lookup failed: {str(e)}")
    PROJECT_ID = "your-fallback-gcp-project-id" # Temporary fallback if session detection slips

LOCATION = "europe-west1" # Jurisdiction of Local Tax Law
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
BUCKET_NAME = f"{PROJECT_ID}-tolleys-source-material"
BLOB_NAME = "revision_source.pdf" # Specific Tolley's file name. If not this then replace with it

vertexai.init(project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client()

def ensure_cloud_infrastructure(local_pdf_path: str) -> str:
    bucket = storage_client.bucket(BUCKET_NAME)
    if not bucket.exists():
        bucket = storage_client.create_bucket(bucket, location=LOCATION)
    blob = bucket.blob(BLOB_NAME)
    if not blob.exists():
        blob.upload_from_filename(local_pdf_path)
    return f"gs://{BUCKET_NAME}/{BLOB_NAME}"


**Scheme to make the Flashcards look nice**

In [ ]:
JSON_SCHEMA = {
    "type": "OBJECT",
    "properties": {
        "flashcards": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "category_tag": {"type": "STRING", "description": "Tax stream, e.g., Corporation Tax, Capital Gains Tax, Stamp Duty"},
                    "statutory_authority": {"type": "STRING", "description": "The exact Act, Schedule, or Section referenced"},
                    "front_prompt": {"type": "STRING", "description": "Clear active-recall revision question targeting a specific rule"},
                    "back_technical_truth": {"type": "STRING", "description": "Precise answer. If calculations, allowances, thresholds, or multiple rules are involved, you MUST format that breakdown as a Markdown table within this text field string."},
                    "visual_layout_hint": {
                        "type": "OBJECT",
                        "properties": {
                            "dominant_color_hex": {"type": "STRING", "description": "A high-visibility UI color hex matched to the tax category"},
                            "mnemonic_anchor": {"type": "STRING", "description": "A high-impact mental image or wordplay device for memory retention"},
                            "structural_shape": {"type": "STRING", "description": "UI layout advice, e.g., '2x2 Grid', 'Timeline Arrow', 'Boolean Decision Tree'"}
                        },
                        "required": ["dominant_color_hex", "mnemonic_anchor", "structural_shape"]
                    }
                },
                "required": ["category_tag", "statutory_authority", "front_prompt", "back_technical_truth", "visual_layout_hint"]
            }
        }
    }
}

**Backend of the Agentic Model relevant to Local Tax Law**

In [ ]:
def run_backend_generation_pipeline(gcs_uri: str, output_path: str):
    user_focus_input = input("\n[Prompt Engine] What specific tax topic or rule do you want to build flashcards for right now?\n> ")
    if not user_focus_input.strip():
        user_focus_input = "Extract primary statutory thresholds, limits, and timelines."

    dynamic_user_prompt = (
        f"Parse the staged Tolley's PDF file. Focus your deep scan and extraction exclusively on: {user_focus_input}. "
        f"Populate the requested JSON structure completely using only facts explicitly stated in this context. "
        f"CRITICAL: If the answer includes any multi-step computational logic, percentages, or threshold tiers, "
        f"you must render that breakdown as a Markdown table within the 'back_technical_truth' text field string."
    )

    model = GenerativeModel("gemini-1.5-pro")
    document_pointer = Part.from_uri(mime_type="application/pdf", uri=gcs_uri)
    system_instruction = (
        "You are an elite educational AI architect and UK Chartered Tax Advisor. "
        "Your mission is to transform wall-of-text Tolley's manuals into highly digestible, visual, schema-enforced flashcards. "
        "Maintain zero tolerance for hallucinations or factual drift away from statutory UK tax legislation values."
    )

    response = model.generate_content(
        [document_pointer, dynamic_user_prompt],
        generation_config=GenerationConfig(
            response_mime_type="application/json",
            response_schema=JSON_SCHEMA,
            temperature=0.1
        ),
        system_instruction=system_instruction
    )

**Frontend of the Fashcards, determining their appearance**

In [ ]:
flashcard_dataset = json.loads(response.text)
with open(output_path, "w") as workspace_file:
        json.dump(flashcard_dataset, workspace_file, indent=4)

def render_html_frontend_visualizer(json_file_path: str):
    with open(json_file_path, "r") as f:
        data = json.load(f)
    cards = data.get("flashcards", [])

**Website format of the Flashcards**

In [ ]:
html_content = """
    <!DOCTYPE html>
    <html>
    <head>
    <meta charset="utf-8">
    <script src="https://jsdelivr.net"></script>
    <style>
        :root { --font-stack: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; }
        body { font-family: var(--font-stack); background-color: #f1f5f9; margin: 0; padding: 20px; }
        .header-container { text-align: center; margin-bottom: 30px; border-bottom: 2px solid #cbd5e1; padding-bottom: 15px; }
        .header-title { color: #0f172a; margin: 0; font-size: 26px; font-weight: 800; }
        .card-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(340px, 1fr)); gap: 25px; }
        .flashcard-container { background-color: transparent; width: 100%; height: 340px; perspective: 1000px; }
        .flashcard-inner { position: relative; width: 100%; height: 100%; transition: transform 0.6s cubic-bezier(0.4, 0, 0.2, 1); transform-style: preserve-3d; box-shadow: 0 10px 15px -3px rgba(0,0,0,0.1); }
        .flashcard-container.is-flipped .flashcard-inner { transform: rotateY(180deg); }
        .card-front, .card-back { position: absolute; width: 100%; height: 100%; -webkit-backface-visibility: hidden; backface-visibility: hidden; border-radius: 14px; box-sizing: border-box; padding: 20px; display: flex; flex-direction: column; }
        .card-front { background-color: #ffffff; color: #1e293b; border: 1px solid #cbd5e1; }
        .card-back { background-color: #0f172a; color: #f8fafc; transform: rotateY(180deg); border: 1px solid #0f172a; }
        .badge-row { display: flex; justify-content: space-between; align-items: center; }
        .category-tag { font-size: 11px; font-weight: 700; text-transform: uppercase; padding: 4px 10px; border-radius: 20px; color: white; }
        .statutory-authority { font-size: 11px; color: #64748b; font-family: monospace; font-weight: 600; }
        .card-back .statutory-authority { color: #94a3b8; }
        .prompt-text { font-size: 16px; font-weight: 700; line-height: 1.5; color: #0f172a; margin: 15px 0; }
        .truth-text { font-size: 13px; line-height: 1.5; color: #e2e8f0; overflow-y: auto; flex-grow: 1; margin-bottom: 10px; }
        .truth-text table { width: 100%; border-collapse: collapse; margin: 10px 0; font-size: 12px; background-color: #1e293b; border-radius: 6px; }
        .truth-text th { background-color: #334155; color: #38bdf8; padding: 6px; text-align: left; font-weight: 700; }
        .truth-text td { padding: 6px; border-bottom: 1px solid #334155; color: #f8fafc; }
        .anchor-box { background-color: #f8fafc; border-left: 4px solid #94a3b8; padding: 8px 12px; font-size: 12px; color: #475569; font-style: italic; border: 1px solid #e2e8f0; border-left: 4px solid #cbd5e1; border-radius: 4px; margin-top: 10px; }
        .card-back .anchor-box { background-color: #1e293b; color: #cbd5e1; }
        .session-tracker-bar { display: flex; gap: 10px; margin-top: 10px; z-index: 10; }
        .track-btn { flex: 1; border: none; padding: 6px 10px; font-size: 11px; font-weight: 700; border-radius: 6px; cursor: pointer; text-transform: uppercase; transition: opacity 0.2s; }
        .track-btn:hover { opacity: 0.9; }
        .btn-mastered { background-color: #10b981; color: white; }
        .btn-review { background-color: #f59e0b; color: white; }
        .flashcard-container.mastered-state { opacity: 0.45; filter: grayscale(40%); }
        .status-banner { font-size: 10px; font-weight: 800; text-transform: uppercase; letter-spacing: 1px; display: none; }
        .flashcard-container.mastered-state .status-banner { display: inline; color: #10b981; }
        .flashcard-container.review-state .status-banner { display: inline; color: #f59e0b; }
        .clickable-zone { flex-grow: 1; display: flex; flex-direction: column; justify-content: flex-start; }
        .layout-footer { font-size: 10px; color: #64748b; text-align: right; font-weight: 700; }
    </style>
    </head>
    <body>
        <div class="header-container">
            <h1 class="header-title">Tolley's Interactive Visualizer</h1>
            <p style="color: #64748b; margin: 5px 0;">Active Session Memory Tracker Enabled • Dynamic Computational Rendering</p>
        </div>
        <div class="card-grid">
    """

for idx, card in enumerate(cards):
    hint = card.get("visual_layout_hint", {})
    color = hint.get("dominant_color_hex", "#3b82f6")
    anchor = hint.get("mnemonic_anchor", "No anchor compiled.")
    shape = hint.get("structural_shape", "Standard Layout")